# Fixed vs. adaptive deposition schemes

When depositing particle data onto a grid, `smudgy` supports two broad
categories of assignment schemes:

- **Fixed (non-adaptive) schemes**: The kernel support size is set
  entirely by the grid spacing, independent of the local particle
  distribution. These correspond to the classical particle-mesh (PM)
  assignment schemes such as Nearest Grid Point (NGP), Cloud-in-Cell (CIC), and Triangular Shaped Cloud (TSC), as well as higher-order piecewise cubic, quartic, and quintic B-splines.
- **Adaptive schemes**: This is the default behaviour in `smudgy`
  (`adaptive=True`), where the kernel support is determined
  per-particle from the local density environment, typically via a
  smoothing length or tensor estimated from the distance to a fixed number of nearest neighbours.

The key practical difference between the two is that fixed schemes require **no smoothing length** at all, since the kernel size is already fully determined by the grid geometry. Adaptive schemes, on the other hand, require an explicit {py:meth}`~smudgy.pointcloud.PointCloud.compute_smoothing` step beforehand, which determines each particle's local kernel size.

In this tutorial we first introduce the fixed schemes, deposit a toy
dataset with each of them, and compare the results side by side. We then
move on to adaptive kernels, and compare how the *kernel* (`kernel_name`) and *structure* (`separable`, `isotropic`, `covariant`, and volume-preserving `covariant`) affect the final result.

First, we import the necessary packages and modules.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import smudgy as sm

np.random.seed(0)

In [2]:
try:
    plt.style.use('custom')
except:
    pass

%matplotlib inline
%config InlineBackend.figure_format='retina'

Next, we initialize a random toy dataset and set up a {py:class}`~smudgy.pointcloud.PointCloud`
object. All particles carry unit weight, and we deposit the weights themselves
(`fields=None`), so every map shows the deposited kernel weights. The
number of neighbours is only needed later, when we compute adaptive
smoothing lengths.

In [3]:
# Define parameters for the point cloud
N = 10
boxsize = 1.0

# Generate random positions for the point cloud (unit weights by default)
positions = np.random.uniform(0.1 * boxsize, 0.9 * boxsize, (N, 2))

# init smudgy object
pc = sm.PointCloud(
    positions=positions,
    boxsize=boxsize,
    verbose=False,
    arch='cpu'
    )

# store fixed kwargs for later use
kwargs = dict(
    fields=None,  # deposit the particle weights only
    averaged=False,
    extent=[[0, boxsize], [0, boxsize]],
)

## Fixed (non-adaptive) deposition schemes

The fixed assignment schemes in `smudgy` are the cardinal B-splines of order $p$ on a grid with cell spacing $s$ (see {doc}`kernels and structures <../../wiki/kernel_structures>`). Order $p$ spans $p$ cells per axis, i.e. $p^d$ cells per particle, and is $C^{p-2}$ continuous:

| Scheme | `kernel_name` | $p$ | Cells/axis | Support | Continuity |
|:--|:-:|:-:|:-:|:-:|:-:|
| Nearest Grid Point (NGP) | `"ngp"` | 1 | 1 | $\pm 0.5\,s$ | discontinuous |
| Cloud-in-Cell (CIC) | `"cic"` | 2 | 2 | $\pm 1.0\,s$ | $C^0$ |
| Triangular Shaped Cloud (TSC) | `"tsc"` | 3 | 3 | $\pm 1.5\,s$ | $C^1$ |
| Cubic B-spline | `"bspline4"` | 4 | 4 | $\pm 2.0\,s$ | $C^2$ |
| Quartic B-spline | `"bspline5"` | 5 | 5 | $\pm 2.5\,s$ | $C^3$ |
| Quintic B-spline | `"bspline6"` | 6 | 6 | $\pm 3.0\,s$ | $C^4$ |

They are selected with `adaptive=False`. Below we compare the resulting deposition grids for all available orders in `smudgy`.

In [4]:
ADAPTIVE = False

gridnums = 20
kernel_names = ['ngp', 'cic', 'tsc', 'bspline4', 'bspline5', 'bspline6']
kernel_labels = [
    'Nearest Grid Point', 
    'Cloud-in-Cell', 
    'Triangular Shaped Cloud', 
    'Cubic B-spline', 
    'Quartic B-spline', 
    'Quintic B-spline'
    ]

fixed_maps = []
for kernel_name in kernel_names:

    weight = pc.deposit(
        **kwargs,
        gridnums=gridnums,
        kernel_name=kernel_name,
        adaptive=ADAPTIVE,
    )

    F = weight
    F = np.where(F == 0, np.nan, F)
    fixed_maps.append(F)

In [5]:
fig, ax = plt.subplots(
    2,
    len(kernel_names) // 2,
    figsize=(4.5, 3.6),
    sharex=True,
    sharey=True,
    layout='constrained'
    )
ax = ax.flat

for j, (kernel_name, label) in enumerate(zip(kernel_names, kernel_labels)):

    F = fixed_maps[j]
    ax[j].pcolormesh(F, vmin=0, vmax=np.nanmax(F))
    ax[j].scatter(positions[:, 1] * gridnums,
                   positions[:, 0] * gridnums,
                   c='r', s=2)

    ax[j].set(xticks=[], yticks=[])
    ax[j].set_aspect('equal')
    ax[j].set_title(label, fontsize=11)

fig.savefig("fixed_methods.png", dpi=500, bbox_inches="tight", transparent=False)
plt.close()

```{image} fixed_methods.png
:width: 100%
:align: center
```

The figure shows the six schemes applied to the same point set (red) and grid, with colours normalized per panel. The stencil widens with increasing order: NGP assigns each point entirely to its host cell, CIC spreads it over $2^d$ cells, TSC over $3^d$, and the higher-order B-splines over correspondingly wider footprints. Because the stencil is set by the grid rather than by the local point density, isolated and clustered points receive identical footprints.

## Adaptive deposition schemes

Adaptive deposition is `smudgy`'s default behaviour (`adaptive=True`). Here, the kernel size (and, for `covariant`, its shape) follows from the smoothing lengths or tensors computed by {py:meth}`~smudgy.pointcloud.PointCloud.compute_smoothing`.

Only kernels with a one-dimensional separable version support all three of the
`separable`, `isotropic`, and `covariant` structures, namely
`tophat`, `tsc`, and `gaussian`. The remaining kernels can only be used
with the `isotropic` and `covariant` structures.

For the `covariant` structure we additionally show the volume-preserving
variant (`volume_preserving=True` in
{py:meth}`~smudgy.pointcloud.PointCloud.compute_smoothing`), which rescales
each smoothing tensor such that its volume matches that of the isotropic
kernel of the same particle, $\det\mathbfss{H}_{\rm vp} = h^d$, i.e. the anisotropy only changes the kernel's
shape and orientation, not its volume.

Below, we deposit the same point set with the four structures (columns) and three example kernels (rows).

In [6]:
ADAPTIVE = True

num_neighbors = 3
gridnums = 512
# (structure, volume_preserving, label)
structures = [
    ('separable', False, 'separable'),
    ('isotropic', False, 'isotropic'),
    ('covariant', False, 'covariant'),
    ('covariant', True, 'covariant-vp'),
]
kernel_names = ['tophat', 'tsc', 'gaussian']

adaptive_maps = {}
for s, (structure, volume_preserving, _) in enumerate(structures):

    pc.global_setup(num_neighbors=num_neighbors, structure=structure)
    pc.compute_smoothing(volume_preserving=volume_preserving)

    for kernel_name in kernel_names:

        weight = pc.deposit(
            **kwargs,
            gridnums=gridnums,
            kernel_name=kernel_name,
            structure=structure,
            adaptive=ADAPTIVE,
        )

        F = weight
        F = np.where(F == 0, np.nan, F)
        adaptive_maps[kernel_name, s] = F

In [7]:
fig, ax = plt.subplots(
    len(kernel_names),
    len(structures),
    figsize=(4, 3),
    sharex=True,
    sharey=True,
    gridspec_kw=dict(wspace=0.04, hspace=0.04),
    )

for i, kernel_name in enumerate(kernel_names):
    for j, (_, _, label) in enumerate(structures):
        F = adaptive_maps[kernel_name, j]
        ax[i, j].pcolormesh(F, vmin=0, vmax=np.nanmax(F), cmap='viridis')
        ax[i, j].scatter(positions[:, 1] * gridnums,
                         positions[:, 0] * gridnums,
                         c='r', s=2)
        ax[i, j].set_aspect('equal')
        ax[i, j].set(xticks=[], yticks=[])
        ax[0, j].set_title(label, fontsize=9)

    ax[i, 0].set_ylabel(kernel_name)

fig.savefig("adaptive.png", dpi=500, bbox_inches="tight", transparent=False)
plt.close()

```{image} adaptive.png
:width: 100%
:align: center
```

Colours are normalized per panel and show the deposited kernel weights. The top-hat kernel (top row) reveals the bare support of each point: axis-aligned boxes for `separable`, circles for `isotropic`, and ellipses aligned with the local principal axes for `covariant`. The remaining rows show the summed field of all kernels. The `covariant` kernels stretch along the local point distribution and thus preserve the filamentary structure better than the `separable` and `isotropic` ones. The volume-preserving variant (`covariant-vp`) has the same kernel areas as the `isotropic` structure and differs from it only in shape and orientation, whereas the default `covariant` kernels are sized to just enclose the neighbours of each point.